# Sentence-Level Argument-Adjunct Asymmetry Analysis

**Subtitle:** Dependency distance minimization (MDD) across spoken vs written Slovenian UD treebanks

This notebook demonstrates a statistical analysis of whether dependency distances differ between spoken and written language across syntactic categories (arguments, adjuncts, modifiers). We'll analyze the complete pipeline:

1. **Load & classify** arcs into syntactic categories
2. **Aggregate** to sentence-level mean dependency distances
3. **Filter** sentences with complete category coverage
4. **Normalize** via log-linear regression on pooled data
5. **Bootstrap** confidence intervals on residuals per category
6. **Compute** asymmetry index (ADJUNCT - ARGUMENT difference)
7. **Visualize** distributions and results

The key finding: **Slovenian shows ARGUMENT shortening in spoken language (CI excludes 0), but ADJUNCT shows no difference — yielding a borderline asymmetry index.**

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages
_pip('loguru==0.7.2')
_pip('psutil==6.1.0')

# Core packages (pre-installed on Colab; install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0', 'scikit-learn==1.6.1')

In [ ]:
import json
import sys
import gc
import math
import resource
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from loguru import logger

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-033e16-argument-adjunct-asymmetry-in-spoken-reg/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    """Load mini demo data from GitHub (with fallback to local)."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local")

In [ ]:
data = load_data()
print(f"Loaded {len(data.get('datasets', []))} dataset(s)")
for ds in data.get('datasets', []):
    print(f"  - {ds.get('dataset')}: {len(ds.get('examples', []))} examples")

## Configuration

Adjust these parameters to scale the analysis. Start with small values to test, then increase.

In [ ]:
# ── Configuration: Start small, scale up ──────────────────────────────────────
RANDOM_SEED = 42
B_BOOTSTRAP = 100  # Start with 100, can increase to 1000 for final run

# For demo: use all 100 examples from mini data (or limit further if testing)
MAX_EXAMPLES = None  # Use all loaded examples

# Category definitions (from original method)
ARGUMENT = {"nsubj", "obj", "iobj", "ccomp", "xcomp", "csubj", "csubj:outer", "nsubj:pass"}
ADJUNCT  = {"advcl", "acl", "acl:relcl", "advcl:relcl"}
MODIFIER_PREFIXES = ("nmod", "amod", "advmod")

LANGUAGE_META = {
    "sl": {"full_name": "Slovenian", "spoken_tb": "sl_sst", "written_tb": "sl_ssj"},
    "fr": {"full_name": "French",    "spoken_tb": "fr_rhapsodie", "written_tb": "fr_gsd"},
}

print(f"Config: B_BOOTSTRAP={B_BOOTSTRAP}, MAX_EXAMPLES={MAX_EXAMPLES}, SEED={RANDOM_SEED}")

## Helper Functions

These are the core building blocks from the original method.py:
- **classify_deprel**: Map dependency relations to syntactic categories
- **fit_log_regression**: OLS fit for log(mdd) ~ log(sent_len)
- **bootstrap_diff**: Unpaired bootstrap on mean differences
- **cohens_d**: Effect size calculation

In [ ]:
def classify_deprel(deprel: str) -> str | None:
    """Classify a dependency relation into ARGUMENT, ADJUNCT, or MODIFIER."""
    if deprel in ARGUMENT:
        return "ARGUMENT"
    if deprel in ADJUNCT:
        return "ADJUNCT"
    if any(deprel == p or deprel.startswith(p + ":") for p in MODIFIER_PREFIXES):
        return "MODIFIER"
    return None


def fit_log_regression(sub: pd.DataFrame) -> dict:
    """Fit log(mdd) ~ log(sent_len) OLS; return slope, intercept, r_sq, residuals array."""
    x = np.log(sub["sent_len"].values.astype(float))
    y = np.log(sub["mdd"].values.astype(float))
    slope, intercept, r, p, se = stats.linregress(x, y)
    residuals = y - (slope * x + intercept)
    return {"slope": slope, "intercept": intercept, "r_sq": r**2, "residuals": residuals}


def bootstrap_diff(spoken_res: np.ndarray, written_res: np.ndarray,
                   b: int = B_BOOTSTRAP, rng: np.random.Generator = None) -> dict:
    """Unpaired bootstrap of mean(spoken_res) - mean(written_res)."""
    if rng is None:
        rng = np.random.default_rng(RANDOM_SEED)
    n_sp, n_wr = len(spoken_res), len(written_res)
    point = float(spoken_res.mean() - written_res.mean())
    diffs = np.empty(b)
    for i in range(b):
        s = rng.choice(spoken_res, n_sp, replace=True).mean()
        w = rng.choice(written_res, n_wr, replace=True).mean()
        diffs[i] = s - w
    ci_lo, ci_hi = float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))
    se = float(diffs.std())
    return {"mean_diff": point, "ci_lower": ci_lo, "ci_upper": ci_hi, "se": se, "boot_dist": diffs}


def cohens_d(a: np.ndarray, b: np.ndarray) -> float:
    """Calculate Cohen's d effect size."""
    n1, n2 = len(a), len(b)
    if n1 + n2 <= 2:
        return float("nan")
    pooled_sd = math.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2))
    if pooled_sd == 0:
        return float("nan")
    return float((a.mean() - b.mean()) / pooled_sd)

## Phase 1: Load & Classify Arcs

Extract arcs from the dataset and classify each by syntactic category.
Input: raw examples with metadata (language, modality, treebank, deprel, distance, sent_len)
Output: DataFrame with one row per arc, labeled with category

In [ ]:
rows = []
unclassified = set()
total = 0

for ds in data["datasets"]:
    for ex in ds["examples"]:
        total += 1
        if MAX_EXAMPLES and total > MAX_EXAMPLES:
            break
        cat = classify_deprel(ex["metadata_deprel"])
        if cat is None:
            unclassified.add(ex["metadata_deprel"])
            continue
        rows.append({
            "language":    ex["metadata_language"],
            "modality":    ex["metadata_modality"],
            "treebank":    ex["metadata_treebank"],
            "sentence_id": ex["metadata_sentence_id"],
            "deprel":      ex["metadata_deprel"],
            "category":    cat,
            "distance":    ex["metadata_dependency_distance"],
            "sent_len":    ex["metadata_sentence_length"],
        })
    if MAX_EXAMPLES and total > MAX_EXAMPLES:
        break

df = pd.DataFrame(rows)
logger.info(f"Phase 1: Loaded {len(df)} classified arcs (total={total}, unclassified={len(unclassified)})")
logger.info(f"Unclassified deprels: {unclassified}")
print(f"\nArc distribution by language/modality/category:")
print(df.groupby(["language", "modality", "category"]).size())

## Phase 2: Sentence-Level Aggregation

Aggregate arcs to mean dependency distance per (sentence, category).
This reduces 100+ arcs per sentence to one row per sentence-category pair.

In [ ]:
sent = df.groupby(["language", "modality", "treebank", "sentence_id", "category"]).agg(
    mdd=("distance", "mean"),
    sent_len=("sent_len", "first"),
    arc_count=("distance", "count"),
).reset_index()

n_before = sent["sentence_id"].nunique()
logger.info(f"Phase 2: Aggregated to {len(sent)} rows ({n_before} unique sentences)")
print(f"\nSentence-level aggregation:")
print(sent.head(10))
del df; gc.collect()

## Phase 3: Filter Complete Sentences

Keep only sentences that have **at least 1 arc in ALL three categories**.
This ensures fair comparison across ARGUMENT, ADJUNCT, and MODIFIER within each sentence.

In [ ]:
cats_per_sent = sent.groupby(["language", "modality", "sentence_id"])["category"].nunique()
complete = cats_per_sent[cats_per_sent == 3].index
mask = pd.MultiIndex.from_arrays([sent["language"], sent["modality"], sent["sentence_id"]]).isin(complete)
sent_filt = sent[mask].copy()
n_after = sent_filt["sentence_id"].nunique()

logger.info(f"Phase 3: Filtered to {n_after} complete sentences (was {cats_per_sent.shape[0]})")
print(f"Filtered: {n_before} → {n_after} sentences (kept {100*n_after/n_before:.1f}%)")
del sent; gc.collect()

## Phase 4: Length Normalization

Fit OLS log(mdd) ~ log(sent_len) **pooled across spoken+written** per language-category pair.
Then compute residuals for each sentence. This removes the effect of sentence length
while **preserving the spoken-written mean difference** in residuals.

In [ ]:
if sent_filt.empty:
    logger.error("No complete sentences after filtering")
    raise ValueError("No complete sentences available.")

reg_stats = {}
residuals_list = []

for (lang, cat), grp in sent_filt.groupby(["language", "category"]):
    res = fit_log_regression(grp)  # pooled across spoken+written
    reg_stats.setdefault(lang, {})[cat] = {
        "slope": float(res["slope"]),
        "intercept": float(res["intercept"]),
        "r_sq": float(res["r_sq"]),
    }
    tmp = grp.copy()
    tmp["residual_mdd"] = res["residuals"]
    residuals_list.append(tmp)

sent_norm = pd.concat(residuals_list, ignore_index=True)
logger.info(f"Phase 4: Computed {len(sent_norm)} normalized rows with residuals")
print("\nLength normalization stats:")
for lang in sorted(reg_stats.keys()):
    print(f"  {lang}:")
    for cat in ["ARGUMENT", "ADJUNCT", "MODIFIER"]:
        if cat in reg_stats[lang]:
            r_sq = reg_stats[lang][cat]["r_sq"]
            print(f"    {cat}: r² = {r_sq:.4f}")
del sent_filt; gc.collect()

## Phase 5: Bootstrap Analysis Per Language

For each language and category, compute:
- **Mean difference** (spoken residuals - written residuals)
- **95% CI** via unpaired bootstrap (B resamples)
- **Cohen's d** effect size with bootstrap CI

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
lang_results = {}

for lang in sent_norm["language"].unique():
    if lang not in LANGUAGE_META:
        logger.warning(f"Unknown language {lang}, skipping")
        continue
    
    sp_all = sent_norm[(sent_norm["language"] == lang) & (sent_norm["modality"] == "spoken")]
    wr_all = sent_norm[(sent_norm["language"] == lang) & (sent_norm["modality"] == "written")]
    n_sp = sp_all["sentence_id"].nunique()
    n_wr = wr_all["sentence_id"].nunique()
    logger.info(f"  {lang}: n_spoken={n_sp}, n_written={n_wr}")
    
    cat_results = {}
    cat_diffs_boot = {}
    
    for cat in ["ARGUMENT", "ADJUNCT", "MODIFIER"]:
        sp_res = sp_all[sp_all["category"] == cat]["residual_mdd"].values
        wr_res = wr_all[wr_all["category"] == cat]["residual_mdd"].values
        
        if len(sp_res) < 2 or len(wr_res) < 2:
            logger.warning(f"  {lang}/{cat}: underpowered (sp={len(sp_res)} wr={len(wr_res)})")
            cat_results[cat] = {"underpowered": True}
            continue
        
        boot = bootstrap_diff(sp_res, wr_res, b=B_BOOTSTRAP, rng=rng)
        cat_diffs_boot[cat] = boot["boot_dist"]
        interp = ("shorter in spoken" if boot["ci_upper"] < 0
                  else ("longer in spoken" if boot["ci_lower"] > 0
                        else "no difference"))
        cat_results[cat] = {
            "mean_diff_residual_mdd": boot["mean_diff"],
            "ci_lower": boot["ci_lower"],
            "ci_upper": boot["ci_upper"],
            "se_bootstrap": boot["se"],
            "n_spoken_sentences": int(len(sp_res)),
            "n_written_sentences": int(len(wr_res)),
            "cohens_d": cohens_d(sp_res, wr_res),
            "interpretation": interp,
        }
    
    # Asymmetry index: Δ_ADJUNCT − Δ_ARGUMENT
    asym_val = float("nan")
    asym_ci_lo = asym_ci_hi = float("nan")
    if "ARGUMENT" in cat_diffs_boot and "ADJUNCT" in cat_diffs_boot:
        arg_dist = cat_diffs_boot["ARGUMENT"]
        adj_dist = cat_diffs_boot["ADJUNCT"]
        asym_val = float(adj_dist.mean() - arg_dist.mean())
        asym_boot = adj_dist - arg_dist
        asym_ci_lo = float(np.percentile(asym_boot, 2.5))
        asym_ci_hi = float(np.percentile(asym_boot, 97.5))
        asym_interp = ("positive" if asym_ci_lo > 0
                       else ("negative" if asym_ci_hi < 0 else "near-zero"))
    else:
        asym_interp = "underpowered"
    
    lang_results[lang] = {
        "language": LANGUAGE_META[lang]["full_name"],
        "language_code": lang,
        "n_sentences_spoken": int(n_sp),
        "n_sentences_written": int(n_wr),
        "categories": cat_results,
        "asymmetry_index": {
            "value": float(asym_val),
            "ci_lower": float(asym_ci_lo),
            "ci_upper": float(asym_ci_hi),
            "interpretation": asym_interp,
        },
    }

logger.info(f"Phase 5: Completed bootstrap analysis for {len(lang_results)} language(s)")

## Results Summary

Display key findings: mean differences per category, effect sizes, asymmetry index, and interpretation.

In [ ]:
print("\n" + "="*70)
print("RESULTS BY LANGUAGE & CATEGORY")
print("="*70)

for lang, res in lang_results.items():
    print(f"\n{lang.upper()} ({res['language']}):")
    print(f"  Spoken: {res['n_sentences_spoken']} sentences | Written: {res['n_sentences_written']} sentences")
    
    for cat in ["ARGUMENT", "ADJUNCT", "MODIFIER"]:
        cr = res["categories"].get(cat, {})
        if cr.get("underpowered"):
            print(f"  {cat}: UNDERPOWERED")
        else:
            md = cr.get("mean_diff_residual_mdd", float("nan"))
            ci_lo = cr.get("ci_lower", float("nan"))
            ci_hi = cr.get("ci_upper", float("nan"))
            d = cr.get("cohens_d", float("nan"))
            interp = cr.get("interpretation", "?")
            excl_zero = (ci_lo > 0) or (ci_hi < 0)
            print(f"  {cat}:")
            print(f"    Δ = {md:+.6f}, CI=[{ci_lo:.6f}, {ci_hi:.6f}], CI excludes 0: {excl_zero}")
            print(f"    Cohen's d = {d:+.4f}, interpretation: {interp}")
    
    ai = res["asymmetry_index"]
    ai_val = ai.get("value", float("nan"))
    ai_lo = ai.get("ci_lower", float("nan"))
    ai_hi = ai.get("ci_upper", float("nan"))
    ai_interp = ai.get("interpretation", "?")
    ai_excl_zero = (ai_lo > 0) or (ai_hi < 0)
    print(f"\n  ASYMMETRY INDEX (Δ_ADJUNCT − Δ_ARGUMENT):")
    print(f"    Value = {ai_val:+.6f}, CI=[{ai_lo:.6f}, {ai_hi:.6f}], CI excludes 0: {ai_excl_zero}")
    print(f"    Interpretation: {ai_interp}")

print("\n" + "="*70)

## Visualization: Mean Differences by Category

Plot the mean differences (Δ = spoken − written) per category with 95% bootstrap CI.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

langs = list(lang_results.keys())
cats = ["ARGUMENT", "ADJUNCT", "MODIFIER"]
colors = {"ARGUMENT": "#e74c3c", "ADJUNCT": "#3498db", "MODIFIER": "#2ecc71"}
x_pos = 0
xticks_pos = []
xticks_labels = []

for i, lang in enumerate(langs):
    res = lang_results[lang]
    for j, cat in enumerate(cats):
        cr = res["categories"].get(cat, {})
        if cr.get("underpowered"):
            continue
        
        md = cr.get("mean_diff_residual_mdd", float("nan"))
        ci_lo = cr.get("ci_lower", float("nan"))
        ci_hi = cr.get("ci_upper", float("nan"))
        
        err_lo = md - ci_lo
        err_hi = ci_hi - md
        
        ax.errorbar(x_pos, md, yerr=[[err_lo], [err_hi]], fmt="o", 
                   capsize=5, markersize=8, linewidth=2, 
                   color=colors[cat], label=cat if i == 0 else "")
        
        xticks_pos.append(x_pos)
        xticks_labels.append(f"{lang}/{cat[:3]}")
        x_pos += 1
    x_pos += 0.5  # space between languages

ax.axhline(0, color="gray", linestyle="--", linewidth=1, alpha=0.5)
ax.set_xticks(xticks_pos)
ax.set_xticklabels(xticks_labels, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Δ (spoken − written) in residual log(MDD)")
ax.set_title("Per-Category Dependency Distance Asymmetry with 95% Bootstrap CI")
ax.grid(axis="y", alpha=0.3)
handles, labels = ax.get_legend_handles_labels()
if handles:
    ax.legend(handles[:len(cats)], labels[:len(cats)], loc="best", fontsize=10)
plt.tight_layout()
plt.savefig("asymmetry_by_category.png", dpi=100)
plt.show()
print("Saved asymmetry_by_category.png")

## Final Table: All Results

Comprehensive summary of all findings.

In [ ]:
import pandas as pd

rows_table = []
for lang, res in lang_results.items():
    for cat in ["ARGUMENT", "ADJUNCT", "MODIFIER"]:
        cr = res["categories"].get(cat, {})
        if cr.get("underpowered"):
            rows_table.append({
                "Language": res["language"],
                "Category": cat,
                "N_spoken": res["n_sentences_spoken"],
                "N_written": res["n_sentences_written"],
                "Mean Diff": "—",
                "95% CI": "—",
                "CI excl 0": "—",
                "Cohen's d": "—",
                "Interp": "underpowered",
            })
        else:
            md = cr.get("mean_diff_residual_mdd", float("nan"))
            ci_lo = cr.get("ci_lower", float("nan"))
            ci_hi = cr.get("ci_upper", float("nan"))
            d = cr.get("cohens_d", float("nan"))
            interp = cr.get("interpretation", "?")
            excl_zero = (ci_lo > 0) or (ci_hi < 0)
            rows_table.append({
                "Language": res["language"],
                "Category": cat,
                "N_spoken": res["n_sentences_spoken"],
                "N_written": res["n_sentences_written"],
                "Mean Diff": f"{md:+.4f}",
                "95% CI": f"[{ci_lo:.4f}, {ci_hi:.4f}]",
                "CI excl 0": "Yes" if excl_zero else "No",
                "Cohen's d": f"{d:+.4f}" if not np.isnan(d) else "—",
                "Interp": interp,
            })

results_df = pd.DataFrame(rows_table)
print("\nPER-CATEGORY RESULTS TABLE:")
print(results_df.to_string(index=False))

# Asymmetry index table
asym_rows = []
for lang, res in lang_results.items():
    ai = res["asymmetry_index"]
    ai_val = ai.get("value", float("nan"))
    ai_lo = ai.get("ci_lower", float("nan"))
    ai_hi = ai.get("ci_upper", float("nan"))
    ai_interp = ai.get("interpretation", "?")
    excl_zero = (ai_lo > 0) or (ai_hi < 0)
    asym_rows.append({
        "Language": res["language"],
        "Asymmetry Index": f"{ai_val:+.4f}",
        "95% CI": f"[{ai_lo:.4f}, {ai_hi:.4f}]",
        "CI excl 0": "Yes" if excl_zero else "No",
        "Interpretation": ai_interp,
    })

asym_df = pd.DataFrame(asym_rows)
print("\n\nASYMMETRY INDEX (Δ_ADJUNCT − Δ_ARGUMENT):")
print(asym_df.to_string(index=False))